# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


**Method: Random Forest classifier**

 each page is labeled `up`, `stable`, or `down` based on its month-over-month change in impressions (±20% threshold). The task is therefore classification.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


**Split: grouped by client, stratified by class**

 All pages of a client go entirely to train or test, because pages of the same client share keywords and site quality, so a page-level split would leak information and inflate the scores. The split is a single train/test split made with `GroupShuffleSplit`, repeated with a different seed until each class (`up`, `stable`, `down`) makes up at least 10% of both sets.



In [2]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SRC_DAILY   = f"{BASE}/fact_content_daily_performance/**/*.parquet"
SRC_CLIENTS = f"{BASE}/dim_clients.parquet"
SRC_CONTENT = f"{BASE}/dim_content.parquet"


FEAT_START, FEAT_END   = "2026-03-01", "2026-03-31"   # الـ features: شهر 3
MID                    = "2026-03-15"                 # آخر يوم في النص الأول
LABEL_START, LABEL_END = "2026-04-01", "2026-04-30"   # الـ label: شهر 4

con.execute(f"""
CREATE OR REPLACE TEMP TABLE dataset AS
WITH daily AS (
    SELECT d.client_hash_id, d.content_hash_id, d.report_date,
           d.gsc_impressions, d.gsc_clicks, d.gsc_sum_position,
           d.ga4_pageviews, d.ga4_sessions, d.ga4_users,
           d.ga4_engaged_sessions, d.ga4_total_engagement_sec,
           COALESCE(d.report_date >= CAST(c.ga4_data_start AS DATE), FALSE) AS ga4_ok
    FROM read_parquet('{SRC_DAILY}', hive_partitioning=true) d
    JOIN content_slice s
      ON s.client_hash_id = d.client_hash_id AND s.content_hash_id = d.content_hash_id
    JOIN read_parquet('{SRC_CLIENTS}') c ON c.client_hash_id = d.client_hash_id
    WHERE d.month IN ('2026-03','2026-04')
      AND d.report_date BETWEEN DATE '{FEAT_START}' AND DATE '{LABEL_END}'
      AND d.report_date >= CAST(c.gsc_data_start AS DATE)
),
agg AS (
    SELECT client_hash_id, content_hash_id,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                         AS f01,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)                 AS f02,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)           AS f05,
      SUM(CASE WHEN ga4_ok THEN ga4_engaged_sessions END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f06,
      SUM(CASE WHEN ga4_ok THEN ga4_total_engagement_sec END) * 1.0
        / NULLIF(SUM(CASE WHEN ga4_ok THEN ga4_sessions END), 0)              AS f07,
      SUM(CASE WHEN ga4_ok THEN ga4_pageviews END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f08,
      SUM(CASE WHEN ga4_ok THEN ga4_users END) * 1.0
        / NULLIF(COUNT(CASE WHEN ga4_ok THEN report_date END), 0)             AS f09
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2
),
halves AS (
    SELECT client_hash_id, content_hash_id,
      (report_date > DATE '{MID}') AS is_h2,
      SUM(gsc_impressions) * 1.0 / COUNT(report_date)                AS avg_impr,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)        AS ctr,
      SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS pos
    FROM daily
    WHERE report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
    GROUP BY 1, 2, 3
    HAVING SUM(gsc_impressions) > 0
),
trend AS (
    SELECT client_hash_id, content_hash_id,
      (MAX(CASE WHEN is_h2 THEN avg_impr END) - MAX(CASE WHEN NOT is_h2 THEN avg_impr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN avg_impr END), 0)       AS f03,
      (MAX(CASE WHEN is_h2 THEN ctr END) - MAX(CASE WHEN NOT is_h2 THEN ctr END))
        / NULLIF(MAX(CASE WHEN NOT is_h2 THEN ctr END), 0)            AS f04,
      MAX(CASE WHEN is_h2 THEN pos END) - MAX(CASE WHEN NOT is_h2 THEN pos END) AS f14
    FROM halves
    GROUP BY 1, 2
),
dim AS (
    SELECT content_hash_id,
      DATE_DIFF('day',
        CAST(CASE WHEN content_updated_date IS NOT NULL
                   AND content_updated_date <= DATE '{FEAT_END}'
                  THEN content_updated_date ELSE content_created_date END AS DATE),
        DATE '{FEAT_END}')                                                   AS f10,
      CASE WHEN keyword_hash_id IS NULL OR keyword_hash_id = '' THEN NULL
           ELSE COUNT(*) OVER (PARTITION BY client_hash_id, keyword_hash_id) - 1
      END                                                                    AS f11,
      CASE WHEN main_intent IS NULL THEN NULL
           ELSE DENSE_RANK() OVER (ORDER BY main_intent) - 1 END             AS f12,
      backlinks                                                              AS f13
    FROM read_parquet('{SRC_CONTENT}')
),
lab AS (
    SELECT client_hash_id, content_hash_id,
      SUM(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{FEAT_START}' AND DATE '{FEAT_END}'
               THEN report_date END), 0)                                     AS avg_base,
      COALESCE(SUM(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN gsc_impressions END) * 1.0
        / NULLIF(COUNT(CASE WHEN report_date BETWEEN DATE '{LABEL_START}' AND DATE '{LABEL_END}'
               THEN report_date END), 0), 0)                                 AS avg_next
    FROM daily
    GROUP BY 1, 2
)
SELECT s.content_hash_id AS content_id,
       s.client_hash_id  AS client_id,
       a.f01, a.f02, t.f03, t.f04, a.f05, a.f06, a.f07, a.f08, a.f09,
       m.f10, m.f11, m.f12, m.f13, t.f14,
       CASE WHEN l.avg_next > 1.2 * l.avg_base THEN 'up'
            WHEN l.avg_next < 0.8 * l.avg_base THEN 'down'
            ELSE 'stable' END AS label
FROM content_slice s
LEFT JOIN agg   a ON a.client_hash_id = s.client_hash_id AND a.content_hash_id = s.content_hash_id
LEFT JOIN trend t ON t.client_hash_id = s.client_hash_id AND t.content_hash_id = s.content_hash_id
LEFT JOIN lab   l ON l.client_hash_id = s.client_hash_id AND l.content_hash_id = s.content_hash_id
LEFT JOIN dim   m ON m.content_hash_id = s.content_hash_id
""")

df = con.execute("SELECT * FROM dataset").df()
print(df.shape)
print(df["label"].value_counts(normalize=True).round(3))

HF token: ··········


CatalogException: Catalog Error: Table with name content_slice does not exist!
Did you mean "sqlite_schema"?

LINE 90: FROM content_slice s
              ^

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.